
# Project 4: Cross Country Scoring
Tren Meckel , Kate Ngo , Faye Desarro <br>
October 30, 2024

In [20]:
import matplotlib.pyplot as plt
import pandas as pd
# from google.colab import drive
# drive.mount('/content/drive')
# file_path = '/content/drive/My Drive/Kate, Faye, and Trin - Project/Datasets/NCAC_XC_2023_Women6k-1.csv'


# Data Wrangling
1. Removing NAs

In the code below, we access the csv file and remove NAs (meaning non-finishing runners with no time). We also skip the columns that contain the name of the dataset

In [21]:
file = pd.read_csv("NCAC_XC_2023_Women6k-1.csv", skiprows=2)
df = pd.DataFrame(file)
df = df.dropna()

2. Converting Time to Floats


In [22]:
# take the minutes
df[["Minute", "Seconds"]] = df["Time"].str.split(":", expand = True)

df["Floats"] = df["Minute"].astype(float) + (df["Seconds"].astype(float))/60
df["Floats"] = df["Floats"].round(1).astype(float)


3. Separating Names

In [23]:
df[["Last", "First"]] = df["Name"].str.split(r",\s", expand = True)

4. Removing teams with less than 5 participants

In [24]:
grouped = df.groupby("Team").count()
grouped['5+'] = False
grouped.loc[grouped['Floats'] >= 5, '5+'] = True
grouped = grouped['5+']
merged_df = pd.merge(df, grouped, on='Team')
merged_df = merged_df[merged_df['5+'] == True]

5. Score by Team

In [25]:
# Count runners per team, only taking teams with at least 5 runners
runner_counts = merged_df.groupby('Team').size()
teams_with_enough_runners = runner_counts[runner_counts >= 5].index

# Filter out teams with fewer than 5 runners
df_scoring = merged_df[merged_df['Team'].isin(teams_with_enough_runners)]

# Rank runners within each team and keep the top 7
df_scoring['RankWithinTeam'] = df_scoring.groupby('Team')['Time'].rank(method='first')

# Keep only the top 7 runners per team
df_scoring = df_scoring[df_scoring['RankWithinTeam'] <= 7]
df_scoring['ScoringPlace'] = df_scoring['Time'].rank(method='first')
df_scoring['ScoringPlace'] = df_scoring['ScoringPlace'].astype(int)

6. Ranking the Teams

In [26]:
# Get the top 5 runners for each team for scoring
team_scores = df_scoring[df_scoring['RankWithinTeam'] <= 5].groupby('Team')['ScoringPlace'].sum().reset_index()
team_scores["Runnerslist"] = df_scoring.groupby('Team')["ScoringPlace"]

# Rename the columns for clarity
team_scores.columns = ['Team', 'Score', "Runnerslist"]

# Add team place based on the score
team_scores['Place'] = team_scores['Score'].rank(method='min')
team_scores = team_scores.sort_values(by='Score')


7. Top Finishers Reporting

In [27]:
# Final report of team scores
team_scores["Place"] = team_scores["Place"].astype(int)
print(team_scores[['Team', 'Place', 'Score']])


            Team  Place  Score
6     Wittenberg      1     36
0         DePauw      2     39
5  Ohio Wesleyan      3     88
4        Oberlin      4     99
7        Wooster      5    129
3         Kenyon      6    156
1        Denison      7    195
2          Hiram      8    255


8. First, Second, and Third Teams all NCAC

In [28]:
# Get top finishers
top_finishers = df_scoring.sort_values(by='ScoringPlace').reset_index(drop=True)
first_team = top_finishers.head(5)
second_team = top_finishers.iloc[5:10]
third_team = top_finishers.iloc[10:15]

# Print honors
print("First Team all NCAC:")
print(first_team[['First', 'Last', 'Team']])
print("\nSecond Team NCAC:")
print(second_team[['First', 'Last', 'Team']])
print("\nThird Team all NCAC:")
print(third_team[['First', 'Last', 'Team']])

First Team all NCAC:
    First       Last        Team
0  Sydney     Khosla  Wittenberg
1    Ella    Webster  Wittenberg
2   Megan       Keys  Wittenberg
3  Sophie     Porter      DePauw
4   Dylan  Kretchmar     Wooster

Second Team NCAC:
   First     Last           Team
5   Emma     Hawk     Wittenberg
6  Grace   Thomas         DePauw
7  Grace   Flores         DePauw
8   Lily  Monnett         DePauw
9    Zoe     Ward  Ohio Wesleyan

Third Team all NCAC:
       First        Last     Team
10  Meredith    Sierpina   DePauw
11    Athena    Theranos  Wooster
12      Sage     Reddish  Oberlin
13     Amzie  Maienbrook   DePauw
14     Eliza    Medearis  Oberlin


9. Top 7 runners from each team

In [29]:
# Filter for top 7 runners and group by Team
scoring_places = (
    df_scoring.loc[df_scoring['RankWithinTeam'] <= 7]  # Only consider top 7 runners
    .groupby('Team', as_index=False)  # Ensure we keep Team as a column in the result
    .agg({'ScoringPlace': lambda x: ' '.join(map(str, x))})  # Concatenate ScoringPlaces into a single string
)

# Rename the column for clarity
scoring_places.columns = ['Team', 'Runners']

# Merge with team scores for final display
final_report = team_scores.merge(scoring_places, on='Team')

# Format runners to include displacing runners (6th and 7th)
final_report['Runners'] = final_report.apply(
    lambda row: ' '.join(row['Runners'].split()[:5]) + ' (' + ' '.join(row['Runners'].split()[5:]) + ')'
    if len(row['Runners'].split()) > 5 else row['Runners'], axis=1
)
final_report["Place"] = final_report["Place"].astype(int)
# Print the final report
print(final_report[['Team', 'Place', 'Score', 'Runners']])

            Team  Place  Score                 Runners
0     Wittenberg      1     36      1 2 3 6 24 (28 31)
1         DePauw      2     39      4 7 8 9 11 (14 19)
2  Ohio Wesleyan      3     88  10 16 18 21 23 (25 26)
3        Oberlin      4     99  13 15 17 22 32 (33 38)
4        Wooster      5    129   5 12 35 37 40 (44 47)
5         Kenyon      6    156  20 27 34 36 39 (41 43)
6        Denison      7    195  29 30 42 46 48 (49 50)
7          Hiram      8    255     45 51 52 53 54 (55)
